# 🗳️ Scraping Presidencia 2026 — v2

| Candidato | X / Twitter | Facebook | TikTok |
|---|---|---|---|
| **Iván Cepeda Castro** | [@IvanCepedaCast](https://x.com/IvanCepedaCast) | [IvanCepedaCastro](https://www.facebook.com/IvanCepedaCastro/) | [@ivancepedacast](https://www.tiktok.com/@ivancepedacast) |
| **Abelardo De la Espriella** | [@ABDELAESPRIELLA](https://x.com/ABDELAESPRIELLA) | [DELAESPRIELLAstyle](https://www.facebook.com/DELAESPRIELLAstyle/) | [@delaespriella_style](https://www.tiktok.com/@delaespriella_style) |

**Cambios respecto a v1:**
- 🐦 Twitter corre en un **subproceso Python independiente** (evita conflicto con el event loop de Jupyter en Windows)
- 🔑 Token de Apify se lee con `dotenv_values` (sin pasar por `os.environ`)

## Celda 1 — Imports y rutas

In [34]:
import sys, os, datetime, subprocess, tempfile
import pandas as pd
from pathlib import Path
from dotenv import dotenv_values   # dotenv_values lee directo del archivo, sin tocar os.environ

# ── Rutas ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR     = Path(os.path.abspath(''))
RAIZ_DIR         = NOTEBOOK_DIR.parent
HERRAMIENTAS_DIR = RAIZ_DIR / 'Herramientas'
OUTPUT_DIR       = NOTEBOOK_DIR / 'resultados'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(HERRAMIENTAS_DIR) not in sys.path:
    sys.path.insert(0, str(HERRAMIENTAS_DIR))

# ── Leer token directamente del archivo .env ─────────────────────────────────
env_path    = HERRAMIENTAS_DIR / '.env'
env_vals    = dotenv_values(env_path)          # dict {'APIFY_TOKEN': '...', ...}
APIFY_TOKEN = env_vals.get('APIFY_TOKEN', '')

# Inyectar en el entorno del proceso actual por si algún scraper llama os.getenv
if APIFY_TOKEN:
    os.environ['APIFY_TOKEN'] = APIFY_TOKEN

print(f'📁 Herramientas: {HERRAMIENTAS_DIR}')
print(f'📁 Resultados:   {OUTPUT_DIR}')
print(f'📄 .env:         {"✅ encontrado" if env_path.exists() else "❌ NO existe"}')
print(f'🔑 APIFY_TOKEN:  {"✅ " + APIFY_TOKEN[:14] + "..." if APIFY_TOKEN else "❌ vacío — revisa Herramientas/.env"}')

📁 Herramientas: c:\Users\juansoag\Downloads\Avances_tesis (modificado)\Herramientas
📁 Resultados:   c:\Users\juansoag\Downloads\Avances_tesis (modificado)\Elecciones presidencia 2026\resultados
📄 .env:         ✅ encontrado
🔑 APIFY_TOKEN:  ✅ apify_api_pbZh...


## Celda 2 — Importar scrapers de TikTok y Facebook

In [35]:
from scrapers.tiktok_scraper   import TikTokScraper
from scrapers.facebook_scraper import FacebookScraper

print('✅ Scrapers importados.')

✅ Scrapers importados.


## Celda 3 — Parámetros

In [36]:
END_DATE   = datetime.date.today().strftime('%Y-%m-%d')
START_DATE = (datetime.date.today() - datetime.timedelta(days=15)).strftime('%Y-%m-%d')
print(f'📅 Período: {START_DATE} → {END_DATE}')

SCRAPE_TWITTER  = True
SCRAPE_TIKTOK   = True
SCRAPE_FACEBOOK = True

MAX_TWEETS   = 200
MAX_TIKTOK   = 100
MAX_FACEBOOK = 100

CANDIDATOS = [
    {
        'id_candidato': 'cepeda_ivan',
        'nombre':       'Iván Cepeda Castro',
        'twitter_user': 'IvanCepedaCast',
        'tiktok_user':  'ivancepedacast',
        'facebook_url': 'https://www.facebook.com/IvanCepedaCastro/',
    },
    {
        'id_candidato': 'delaespriella_abelardo',
        'nombre':       'Abelardo De la Espriella',
        'twitter_user': 'ABDELAESPRIELLA',
        'tiktok_user':  'delaespriella_style',
        'facebook_url': 'https://www.facebook.com/DELAESPRIELLAstyle/',
    },
]

for c in CANDIDATOS:
    print(f"  • {c['nombre']}")

📅 Período: 2026-06-06 → 2026-06-21
  • Iván Cepeda Castro
  • Abelardo De la Espriella


## Celda 4 — Funciones auxiliares

### `scrape_twitter_subprocess`
Genera un script Python temporal y lo ejecuta como **proceso separado**.
Así Playwright crea su propio `ProactorEventLoop` limpio, sin conflicto con Jupyter.

In [37]:
# Plantilla del script temporal que correrá Twitter en un proceso propio
_TW_SCRIPT = '''\
import sys, asyncio, os
from pathlib import Path
from dotenv import dotenv_values

HERRAMIENTAS_DIR = Path(r"{herramientas_dir}")
sys.path.insert(0, str(HERRAMIENTAS_DIR))

env = dotenv_values(HERRAMIENTAS_DIR / ".env")
os.environ.update({{k: v for k, v in env.items() if v}})

from scrapers.twitter_scraper import TwitterPlaywrightScraper

async def main():
    scraper = TwitterPlaywrightScraper(headless=False)
    df = await scraper.run_scraper({accounts}, "{start_date}", "{end_date}")
    if not df.empty:
        df.to_csv(r"{output_path}", index=False, encoding="utf-8-sig", sep=";")
        print(f"OK:{{len(df)}}")
    else:
        print("EMPTY")

asyncio.run(main())
'''

def scrape_twitter_subprocess(accounts: list, start_date: str, end_date: str,
                               output_path: Path) -> pd.DataFrame:
    """Ejecuta el scraper de Twitter en un subproceso Python independiente."""
    script_content = _TW_SCRIPT.format(
        herramientas_dir=str(HERRAMIENTAS_DIR).replace('\\', '/'),
        accounts=accounts,
        start_date=start_date,
        end_date=end_date,
        output_path=str(output_path).replace('\\', '/'),
    )
    # Guardar script temporal
    tmp = Path(tempfile.mktemp(suffix='.py'))
    tmp.write_text(script_content, encoding='utf-8')

    print(f'    ⚙️  Lanzando subproceso Twitter para {accounts}...')
    result = subprocess.run(
        [sys.executable, str(tmp)],
        text=True, timeout=600
    )
    tmp.unlink(missing_ok=True)   # limpiar script temporal

    if result.returncode != 0:
        raise RuntimeError(f'El subproceso de Twitter terminó con código {result.returncode}')

    if output_path.exists():
        return pd.read_csv(output_path, sep=';', encoding='utf-8-sig')
    return pd.DataFrame()


def save_append(df: pd.DataFrame, filepath: Path) -> None:
    """Guarda df en CSV modo append. Crea el archivo si no existe."""
    filepath = Path(filepath)
    try:
        if df.empty:
            print(f'    ⚠️  Sin datos para {filepath.name}')
            return
        if filepath.is_file():
            cols = pd.read_csv(filepath, sep=';', nrows=0, encoding='utf-8-sig').columns.tolist()
            df.reindex(columns=cols).to_csv(
                filepath, mode='a', index=False, header=False, encoding='utf-8-sig', sep=';'
            )
        else:
            df.to_csv(filepath, index=False, encoding='utf-8-sig', sep=';')
        print(f'    💾 {filepath.name}  ({len(df)} filas guardadas)')
    except Exception as e:
        print(f'    ❌ Error guardando {filepath.name}: {e}')


print('✅ Funciones auxiliares listas.')

✅ Funciones auxiliares listas.


## Celda 5 — Inicializar scrapers de Apify

In [38]:
# Token pasado explícitamente — no depende de os.getenv ni de la caché del módulo
tt_scraper = TikTokScraper(token=APIFY_TOKEN)
fb_scraper = FacebookScraper(token=APIFY_TOKEN)

print(f'✅ TikTokScraper   token: {tt_scraper.token[:14]}...')
print(f'✅ FacebookScraper token: {fb_scraper.token[:14]}...')

✅ TikTokScraper   token: apify_api_pbZh...
✅ FacebookScraper token: apify_api_pbZh...


## Celda 6 — Scraping principal

> **Twitter**: se abrirá una ventana de Chromium. Si pide login, inicia sesión en X y el proceso continuará solo. La sesión queda guardada en `Herramientas/scrapers/user_data/`.

In [39]:
reporte = []

for idx, cand in enumerate(CANDIDATOS, 1):
    print(f"\n{'='*60}")
    print(f"[{idx}/{len(CANDIDATOS)}] {cand['nombre']}")
    print(f"{'='*60}")

    estado = {
        'id_candidato':    cand['id_candidato'],
        'nombre':          cand['nombre'],
        'twitter_status':  'No aplica' if not cand.get('twitter_user')  else 'Pendiente',
        'twitter_count':   0,
        'tiktok_status':   'No aplica' if not cand.get('tiktok_user')   else 'Pendiente',
        'tiktok_count':    0,
        'facebook_status': 'No aplica' if not cand.get('facebook_url')  else 'Pendiente',
        'facebook_count':  0,
        'timestamp':       datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    }

    # ── Twitter (subproceso) ──────────────────────────────────────────────────
    if SCRAPE_TWITTER and cand.get('twitter_user'):
        print(f"\n🐦 Twitter → @{cand['twitter_user']}")
        try:
            tw_out = OUTPUT_DIR / f"_tw_tmp_{cand['id_candidato']}.csv"
            df_tw  = scrape_twitter_subprocess(
                [cand['twitter_user']], START_DATE, END_DATE, tw_out
            )
            if not df_tw.empty:
                df_tw.insert(0, 'id_candidato', cand['id_candidato'])
                df_tw.insert(1, 'nombre', cand['nombre'])
                save_append(df_tw, OUTPUT_DIR / 'presidencia2026_tweets.csv')
                tw_out.unlink(missing_ok=True)   # limpiar temporal
                estado['twitter_status'] = 'OK'
                estado['twitter_count']  = len(df_tw)
            else:
                estado['twitter_status'] = 'Sin resultados'
        except Exception as e:
            print(f'  ❌ Twitter: {e}')
            estado['twitter_status'] = f'ERROR: {str(e)[:80]}'

    # ── TikTok ────────────────────────────────────────────────────────────────
    if SCRAPE_TIKTOK and cand.get('tiktok_user'):
        print(f"\n🎵 TikTok → @{cand['tiktok_user']}")
        try:
            df_tt = tt_scraper.fetch_profile_data(
                [cand['tiktok_user']], max_items=MAX_TIKTOK,
                start_date=START_DATE, end_date=END_DATE
            )
            if not df_tt.empty:
                df_tt.insert(0, 'id_candidato', cand['id_candidato'])
                df_tt.insert(1, 'nombre', cand['nombre'])
                save_append(df_tt, OUTPUT_DIR / 'presidencia2026_tiktok.csv')
                estado['tiktok_status'] = 'OK'
                estado['tiktok_count']  = len(df_tt)
            else:
                estado['tiktok_status'] = 'Sin resultados'
        except Exception as e:
            print(f'  ❌ TikTok: {e}')
            estado['tiktok_status'] = f'ERROR: {str(e)[:80]}'

    # ── Facebook ──────────────────────────────────────────────────────────────
    if SCRAPE_FACEBOOK and cand.get('facebook_url'):
        print(f"\n📘 Facebook → {cand['facebook_url']}")
        try:
            df_fb = fb_scraper.process_account(
                cand['facebook_url'], START_DATE, END_DATE,
                max_posts=MAX_FACEBOOK
            )
            if not df_fb.empty:
                df_fb.insert(0, 'id_candidato', cand['id_candidato'])
                df_fb.insert(1, 'nombre', cand['nombre'])
                save_append(df_fb, OUTPUT_DIR / 'presidencia2026_facebook.csv')
                estado['facebook_status'] = 'OK'
                estado['facebook_count']  = len(df_fb)
            else:
                estado['facebook_status'] = 'Sin resultados'
        except Exception as e:
            print(f'  ❌ Facebook: {e}')
            estado['facebook_status'] = f'ERROR: {str(e)[:80]}'

    reporte.append(estado)
    pd.DataFrame(reporte).to_csv(
        OUTPUT_DIR / 'presidencia2026_reporte_proceso.csv',
        index=False, sep=';', encoding='utf-8-sig'
    )
    print('\n📋 Reporte parcial guardado.')

print(f"\n{'='*60}\n🏁 PROCESO FINALIZADO\n{'='*60}")


[1/2] Iván Cepeda Castro

🐦 Twitter → @IvanCepedaCast
    ⚙️  Lanzando subproceso Twitter para ['IvanCepedaCast']...

🎵 TikTok → @ivancepedacast
🚀 Iniciando scraper de TikTok para: ['ivancepedacast']
📅 Rango solicitado: 2026-06-06 al 2026-06-21


[apify.tiktok-profile-scraper runId:P4YqaxRDcMUWgTKvJ] -> Status: RUNNING, Message: 
[apify.tiktok-profile-scraper runId:P4YqaxRDcMUWgTKvJ] -> 2026-06-22T00:42:27.715Z ACTOR: Pulling container image of build VOhsioU74CedZIcYF from registry.
[apify.tiktok-profile-scraper runId:P4YqaxRDcMUWgTKvJ] -> 2026-06-22T00:42:27.719Z ACTOR: Creating container.
[apify.tiktok-profile-scraper runId:P4YqaxRDcMUWgTKvJ] -> 2026-06-22T00:42:27.778Z ACTOR: Starting container.
[apify.tiktok-profile-scraper runId:P4YqaxRDcMUWgTKvJ] -> 2026-06-22T00:42:27.780Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.tiktok-profile-scraper runId:P4YqaxRDcMUWgTKvJ] -> 2026-06-22T00:42:29.453Z INFO  System info {"apifyVersion":"3.4.2","apifyClientVersion":"2.12.4","crawleeVersion":"3.13.5","osType":"Linux","nodeVersion":"v22.22.3"}
[apify.tiktok-profile-scraper runId:P4YqaxRDcMUWgTKvJ] -> 2026-06-22T00:42:29.500Z INFO  PHASE -- STARTING ACTOR.
[apify.tiktok-profile-scraper runId:P4YqaxRDcMUWgTKvJ] -> 2026-06-22T00:42

✅ Proceso completado en Apify. Descargando resultados...
    💾 presidencia2026_tiktok.csv  (77 filas guardadas)

📘 Facebook → https://www.facebook.com/IvanCepedaCastro/

🚀 Iniciando Apify Facebook Scraper para: https://www.facebook.com/IvanCepedaCastro/
📅 Rango deseado: 2026-06-06 al 2026-06-21


[apify.facebook-posts-scraper runId:1WzSt1ZPJyc6UaiPI] -> Status: RUNNING, Message: 
[apify.facebook-posts-scraper runId:1WzSt1ZPJyc6UaiPI] -> 2026-06-22T00:42:56.934Z ACTOR: Pulling container image of build LqbgjinlOYd9Cr1QU from registry.
[apify.facebook-posts-scraper runId:1WzSt1ZPJyc6UaiPI] -> 2026-06-22T00:42:56.936Z ACTOR: Creating container.
[apify.facebook-posts-scraper runId:1WzSt1ZPJyc6UaiPI] -> 2026-06-22T00:42:58.839Z ACTOR: Starting container.
[apify.facebook-posts-scraper runId:1WzSt1ZPJyc6UaiPI] -> 2026-06-22T00:42:58.841Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.facebook-posts-scraper runId:1WzSt1ZPJyc6UaiPI] -> 2026-06-22T00:43:01.497Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.19.0","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.16.0"}
[apify.facebook-posts-scraper runId:1WzSt1ZPJyc6UaiPI] -> 2026-06-22T00:43:01.516Z (node:8) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors tha

✅ Proceso completado en Apify para Facebook. Descargando dataset...
📊 100 posts de Facebook obtenidos.
    💾 presidencia2026_facebook.csv  (100 filas guardadas)

📋 Reporte parcial guardado.

[2/2] Abelardo De la Espriella

🐦 Twitter → @ABDELAESPRIELLA
    ⚙️  Lanzando subproceso Twitter para ['ABDELAESPRIELLA']...
    💾 presidencia2026_tweets.csv  (5 filas guardadas)

🎵 TikTok → @delaespriella_style
🚀 Iniciando scraper de TikTok para: ['delaespriella_style']
📅 Rango solicitado: 2026-06-06 al 2026-06-21


[apify.tiktok-profile-scraper runId:zTcTWqSkk0siYPIGK] -> Status: RUNNING, Message: 
[apify.tiktok-profile-scraper runId:zTcTWqSkk0siYPIGK] -> 2026-06-22T00:46:48.751Z ACTOR: Pulling container image of build VOhsioU74CedZIcYF from registry.
[apify.tiktok-profile-scraper runId:zTcTWqSkk0siYPIGK] -> 2026-06-22T00:46:48.753Z ACTOR: Creating container.
[apify.tiktok-profile-scraper runId:zTcTWqSkk0siYPIGK] -> 2026-06-22T00:46:48.933Z ACTOR: Starting container.
[apify.tiktok-profile-scraper runId:zTcTWqSkk0siYPIGK] -> 2026-06-22T00:46:48.934Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.tiktok-profile-scraper runId:zTcTWqSkk0siYPIGK] -> 2026-06-22T00:46:51.233Z INFO  System info {"apifyVersion":"3.4.2","apifyClientVersion":"2.12.4","crawleeVersion":"3.13.5","osType":"Linux","nodeVersion":"v22.22.3"}
[apify.tiktok-profile-scraper runId:zTcTWqSkk0siYPIGK] -> 2026-06-22T00:46:51.299Z INFO  PHASE -- STARTING ACTOR.
[apify.tiktok-profile-scraper runId:zTcTWqSkk0siYPIGK] -> 2026-06-22T00:46

✅ Proceso completado en Apify. Descargando resultados...
    💾 presidencia2026_tiktok.csv  (54 filas guardadas)

📘 Facebook → https://www.facebook.com/DELAESPRIELLAstyle/

🚀 Iniciando Apify Facebook Scraper para: https://www.facebook.com/DELAESPRIELLAstyle/
📅 Rango deseado: 2026-06-06 al 2026-06-21


[apify.facebook-posts-scraper runId:LU1HFMgexI27966Da] -> Status: RUNNING, Message: 
[apify.facebook-posts-scraper runId:LU1HFMgexI27966Da] -> 2026-06-22T00:47:12.098Z ACTOR: Pulling container image of build LqbgjinlOYd9Cr1QU from registry.
[apify.facebook-posts-scraper runId:LU1HFMgexI27966Da] -> 2026-06-22T00:47:12.100Z ACTOR: Creating container.
[apify.facebook-posts-scraper runId:LU1HFMgexI27966Da] -> 2026-06-22T00:47:12.143Z ACTOR: Starting container.
[apify.facebook-posts-scraper runId:LU1HFMgexI27966Da] -> 2026-06-22T00:47:12.144Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.facebook-posts-scraper runId:LU1HFMgexI27966Da] -> 2026-06-22T00:47:12.860Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.19.0","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.16.0"}
[apify.facebook-posts-scraper runId:LU1HFMgexI27966Da] -> 2026-06-22T00:47:12.882Z (node:7) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors tha

✅ Proceso completado en Apify para Facebook. Descargando dataset...
📊 85 posts de Facebook obtenidos.
    💾 presidencia2026_facebook.csv  (85 filas guardadas)

📋 Reporte parcial guardado.

🏁 PROCESO FINALIZADO


## Celda 7 — Resumen

In [40]:
df_reporte = pd.DataFrame(reporte)

for _, row in df_reporte.iterrows():
    print(f"\n👤 {row['nombre']}")
    print(f"   🐦 Twitter:  {row['twitter_status']}  ({row['twitter_count']} tweets)")
    print(f"   🎵 TikTok:   {row['tiktok_status']}   ({row['tiktok_count']} videos)")
    print(f"   📘 Facebook: {row['facebook_status']} ({row['facebook_count']} posts)")

print('\n' + '─' * 40)
for net in ['twitter', 'tiktok', 'facebook']:
    total = df_reporte[f'{net}_count'].sum()
    print(f"  {net.capitalize():10s}: {total} publicaciones")

print(f'\n💾 Archivos en: {OUTPUT_DIR}')
display(df_reporte)


👤 Iván Cepeda Castro
   🐦 Twitter:  Sin resultados  (0 tweets)
   🎵 TikTok:   OK   (77 videos)
   📘 Facebook: OK (100 posts)

👤 Abelardo De la Espriella
   🐦 Twitter:  OK  (5 tweets)
   🎵 TikTok:   OK   (54 videos)
   📘 Facebook: OK (85 posts)

────────────────────────────────────────
  Twitter   : 5 publicaciones
  Tiktok    : 131 publicaciones
  Facebook  : 185 publicaciones

💾 Archivos en: c:\Users\juansoag\Downloads\Avances_tesis (modificado)\Elecciones presidencia 2026\resultados


,id_candidato,nombre,twitter_status,twitter_count,tiktok_status,tiktok_count,facebook_status,facebook_count,timestamp
0,cepeda_ivan,Iván Cepeda Castro,Sin resultados,0,OK,77,OK,100,2026-06-21 19:42:10
1,delaespriella_abelardo,Abelardo De la Espriella,OK,5,OK,54,OK,85,2026-06-21 19:46:25


## Celda 8 — Vista previa

In [41]:
for emoji, archivo in [
    ('🐦 Tweets',   'presidencia2026_tweets.csv'),
    ('🎵 TikTok',   'presidencia2026_tiktok.csv'),
    ('📘 Facebook', 'presidencia2026_facebook.csv'),
]:
    p = OUTPUT_DIR / archivo
    if p.exists():
        df = pd.read_csv(p, sep=';', encoding='utf-8-sig')
        print(f'{emoji}: {len(df)} registros')
        display(df.head(3))
    else:
        print(f'{emoji}: sin datos aún.')

# **Consolidación**

## Celda 9 — Con